# Stabilization Kinematics — Complete-State Feedback

Visualizes hand trajectories during stabilization for the selected target conditions.

## 1. Imports

In [ ]:
import numpy as np 
import matplotlib.pyplot as plt 
import torch 
import ipdb 
from torch.autograd.functional import jacobian 
torch.autograd.set_detect_anomaly(True) 


## 2. Load the Target Table

Load all eight targets in table order.

In [ ]:
target_theta_traj= np.load('../final_theta_traj.npy', allow_pickle= True)
print(target_theta_traj.shape)  


## 3. Load Saved Joint Trajectories

Read the 16 stabilization outputs from the shared data root.

In [ ]:

theta1_traj= []
theta2_traj= []

for jid in range(8):
    theta1j= np.load(f'../stabilization/theta1_{jid+1}.npy')
    theta2j= np.load(f'../stabilization/theta2_{jid+1}.npy')

    theta1_traj.append(theta1j)
    theta2_traj.append(theta2j)
    

## 4. Apply the Target Ordering

In [ ]:


target_theta_traj= np.array(target_theta_traj, copy=True)
theta1_traj= np.array(theta1_traj).tolist()
theta2_traj= np.array(theta2_traj).tolist()


## 5. Convert Joint Angles to Hand Position

Convert joint angles to hand position with two-link forward kinematics.

In [ ]:

l1= 0.15
l2= 0.21


theta1= 0.1
theta2= 1.57

x1= l1*np.cos(theta1)
y1= l1*np.sin(theta1)

x2_initial= x1 + l2*np.cos(theta1+theta2)
y2_initial= y1 + l2*np.sin(theta1+theta2)

x2_cum= []
y2_cum= []

x2f_cum= []
y2f_cum= []

for jid in range(len(target_theta_traj)):

    theta1f= float(target_theta_traj[jid][0])
    theta2f= float(target_theta_traj[jid][1])
    
    x1= l1*np.cos(theta1f)
    y1= l1*np.sin(theta1f)
    
    x2_f= x1 + l2*np.cos(theta1f+theta2f)
    y2_f= y1 + l2*np.sin(theta1f+theta2f)

    x2f_cum.append(x2_f)
    y2f_cum.append(y2_f)

    
    theta1_traj_j= theta1_traj[jid]  
    theta2_traj_j= theta2_traj[jid]

    x2_j= []
    y2_j= []

    for tstep in range(len(theta1_traj_j)):
        
        theta1= theta1_traj_j[tstep]
        theta2= theta2_traj_j[tstep]
        
        x1= l1*np.cos(theta1)
        y1= l1*np.sin(theta1)
        
        x2= x1 + l2*np.cos(theta1+theta2)
        y2= y1 + l2*np.sin(theta1+theta2)

        x2_j.append(x2)
        y2_j.append(y2)

    x2_cum.append(x2_j)
    y2_cum.append(y2_j)


## 6. Plot Stabilization Trajectories

Show target locations, the initial hand position, and the final stabilization samples.

In [ ]:
plt.figure(figsize= (10, 10))

n = len(target_theta_traj)
colors = plt.cm.RdBu(np.linspace(0,1,n))

for job_id in range(len(target_theta_traj)):
    plt.plot(x2_cum[job_id], y2_cum[job_id], color= colors[job_id])

for job_id in range(len(target_theta_traj)):
    plt.scatter(x2f_cum[job_id], y2f_cum[job_id], color= 'grey', s= 20)

plt.scatter(x2_initial, y2_initial, color= 'cyan', s= 80)


for job_id in range(len(target_theta_traj)):
    plt.scatter(x2_cum[job_id][-1], y2_cum[job_id][-1], color= 'red', s= 20)



plt.show()
